# JK Assistance - sample tests

This notebook exercises every part of JK Assistance with sample inputs:

1. Knowledge base overview
2. Procedures: degree, transcript, migration, clearance (with versioned fees)
3. Admission advisor for students after 12 years of education
4. Semester promotion engine (probation, summer waiver, drop, relegation) and the GPA worked examples
5. What-if simulation
6. Proactive student dashboards (GUIDE / ASK_INFO / ALERT / ESCALATE)
7. Event-driven monitoring (a payment clears an alert)
8. Application tracking with a relative deadline
9. The chat pipeline on many sample questions
10. Evidence search and the human-advisor queue
11. The MCP server, called through the official client
12. (Optional) Claude-written answers, if `ANTHROPIC_API_KEY` is set

It runs on an **isolated temporary copy** of the fictional demo data, with "today" fixed at **2026-09-29**, so results are reproducible and `data/` is never modified.

In [1]:
import os, sys, subprocess, tempfile, json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ["JK_TODAY"] = "2026-09-29"
os.environ["JK_DATA_DIR"] = tempfile.mkdtemp(prefix="jk-notebook-")
USE_CLAUDE = bool(os.environ.get("ANTHROPIC_API_KEY"))
if not USE_CLAUDE:
    os.environ["JK_DISABLE_LLM"] = "1"
sys.path.insert(0, str(ROOT))
subprocess.run([sys.executable, str(ROOT / "scripts" / "seed_demo.py")], check=True, env=os.environ, capture_output=True)

import pandas as pd
from IPython.display import Markdown, display
pd.set_option("display.max_colwidth", 90)

from jk_assistance import advisor, applications, assistant, monitor, progression, store
from jk_assistance.decision import triage
from jk_assistance.knowledge import get_knowledge

kb = get_knowledge()
print("Project:", ROOT)
print("Claude answers:", "ON" if USE_CLAUDE else "OFF (rule-engine rendering)")

Project: C:\Users\Faisal Ramzan\Desktop\AJK university VA
Claude answers: OFF (rule-engine rendering)


## 1. Knowledge base overview

In [2]:
print(f"{len(kb.institutions)} institutions, {len(kb.services)} services, {len(kb.programs)} admission programs, "
      f"{len(kb.dated_deadlines)} dated deadlines, {len(kb.relative_rules)} deadline rules, {len(kb.chunks)} evidence chunks")
pd.DataFrame([{"institution": s["institution"], "service": s["name"], "fees": len(s.get("fees", [])),
               "documents": len(s.get("required_documents", [])), "trust": s.get("trust")} for s in kb.services.values()])

7 institutions, 27 services, 38 admission programs, 13 dated deadlines, 12 deadline rules, 1560 evidence chunks


,institution,service,fees,documents,trust
0,uajk,Degree issuance (original degree),7,4,official
1,uajk,Verification of Degree / DMC / Result Intimation / Merit or Position certificate / Gol...,1,3,official
2,uajk,Duplicate degree (lost / damaged degree),1,6,official
3,uajk,"Correction / revised degree, and change of name or father's name in university record",6,3,official
4,uajk,"Detailed Marks Certificate (DMC) / transcript, semester result card, percentage certif...",6,1,official
5,uajk,Migration certificate / NOC (leaving UAJK for another university),2,6,official
6,uajk,"Transfer / migration between universities or colleges (university-to-university, colle...",1,5,official
7,uajk,Bonafide certificate,1,4,official
8,uajk,English proficiency certificate (medium of instruction letter),1,3,official
9,uajk,Re-checking of answer book / result,2,2,official


In [3]:
pd.Series([c.kind for c in kb.chunks]).value_counts().rename("evidence chunks by kind").to_frame()

,evidence chunks by kind
document,1317
page,135
program,46
service,27
deadline,25
institution,7
rule,3


## 2. Procedures - checklists with the fee in force today

In [4]:
r = assistant.answer("How do I get my degree from MUST?")
print(r["intent"], r["action"]); display(Markdown(r["answer"]))

service GUIDE


### Degree issuance (original degree) - Mirpur University of Science and Technology (MUST)
**Office:** Controller of Examinations, Admin Block, Jarikas Campus, Mirpur (05827-961038 / 961068 / 961065, controller@must.edu.pk)

**Fees (version in force today):**
- Degree before convocation (including form fee): **Rs. 9,080** - effective 2026-07-28; previously Rs. 7,500 from 2024-09-02
- Degree after convocation (including form fee): **Rs. 3,270** - effective 2026-07-28; previously Rs. 2,700 from 2024-09-02
- Urgent degree on declaration of special result (including form fee): **Rs. 22,840** - effective 2026-07-28; previously Rs. 19,350 from 2024-09-02
- Degree under process letter / degree completion letter: **Rs. 1,210** - effective 2026-07-28; previously Rs. 1,100 from 2024-09-02
- Merit certificate: **Rs. 3,810** - effective 2026-07-28
- Duplicate merit certificate: **Rs. 7,610** - effective 2026-07-28

**Documents to attach:**
- [ ] Completed Degree Form (names in English block letters and Urdu, CNIC, registration no., session, marks/CGPA)
- [ ] Attested copy of Secondary School Certificate (SSC)
- [ ] Attested copy of Detailed Marks Certificate (DMC)
- [ ] Attested copy of CNIC
- [ ] Clearance Certificate from the department (regular university students only) - see must.clearance
- [ ] Original internship letter of minimum 3 months (compulsory for B.Com and ADC annual-system students)
- [ ] Paid HBL/ABL challan or bank draft (amount in digits and words on the form)
- [ ] Attestation by Dean / Chairman / Director / Coordinator / Principal (regular/college) or Gazetted Officer Class-I (external/private)

**Steps:**
1. Get your departmental Clearance Certificate signed by all offices (must.clearance).
2. Pay the degree fee for your case (before/after convocation or urgent).
3. Fill the Degree Form, get it attested and attach the documents.
4. Submit to the Controller of Examinations. The degree is posted to the postal address on the form (not the CNIC address).

**Processing time:** About 2 months from receipt of a complete application; urgent degree about 20-35 days. Incomplete forms are not entertained.

**Forms:** [Degree Form (2026 update, scanned)](https://must.edu.pk/wp-content/uploads/2026/08/Degree-from-updates.pdf)

**Notes:** If you cannot collect it yourself, an authorised person may collect the degree with an original affidavit on stamp paper and the degree holder's original CNIC. Fee versions - the 28 July 2026 schedule applies "to all sessions from 28 July 2026"; earlier payments followed the 2 Sept 2024 schedule.

_Sources: https://must.edu.pk/wp-content/uploads/2026/08/Degree-from-updates.pdf, https://must.edu.pk/wp-content/uploads/2024/09/Miscellaneous-Fee-Structure-2024.pdf_

In [5]:
display(Markdown(assistant.answer("UAJK migration certificate documents and fee")["answer"]))

### Migration certificate / NOC (leaving UAJK for another university) - The University of Azad Jammu & Kashmir (UAJK)
**Office:** Assistant Registrar (General), General Section, Admin Block Chellah Campus (05822-960425)

**Fees (version in force today):**
- Migration certificate (ordinary, within 2 weeks): **Rs. 5,800** - effective 2025-10-01
- Duplicate migration certificate (within 2 weeks): **Rs. 17,700** - effective 2025-10-01

**Documents to attach:**
- [ ] Migration Certificate Form, attested by the principal/head of institution (regular students) or a first-class Gazetted Officer (private candidates)
- [ ] Original HBL challan pasted on the form at the marked place (deposited in UAJK's normal account; challans for another university's account are not accepted). Overseas candidates may attach an original HBL bank draft in favour of Director Finance & Planning, UAJK.
- [ ] Attested clear copies of the last result card / degree available
- [ ] Attested copy of CNIC
- [ ] If attaching a fail result card and remaining chances were not availed - affidavit on stamp paper attested by an oath commissioner
- [ ] For duplicate migration (within one year of the first) - FIR, two original national newspaper advertisements, and a court-attested affidavit that the first certificate was lost and not used

**Steps:**
1. Pay the fee in UAJK's normal account through an HBL challan.
2. Fill and get the form attested; paste the challan; attach documents.
3. Send the case within 15 days of paying the fee, addressed to the Assistant Registrar (General), UAJK Muzaffarabad.

**Deadline:** Submit the case within 15 days after depositing the fee

**Forms:** [Migration Certificate Form](https://uajk.edu.pk/wp-content/uploads/2025/10/Migration-form-up-dated.pdf)

_Sources: https://uajk.edu.pk/wp-content/uploads/2025/10/Migration-form-up-dated.pdf, https://challan.uajk.edu.pk/_

In [6]:
display(Markdown(assistant.answer("MBBS transcript from UHS")["answer"]))

### MBBS / BDS transcript, transcript verification, duplicate DMC - University of Health Sciences (UHS), Lahore - degree-awarding university for AJK public medical colleges
**Office:** UHS Examination Cell (042-99231304-9 Ext. 356, 042-99231075, tab@uhs.edu.pk)
**Portal:** https://dip.uhs.edu.pk/

**Fees (version in force today):**
- Issuance of transcript (per transcript): **Rs. 2,000** (urgent Rs. 4,000) - as fetched on 2026-09-29
- Issuance of duplicate DMC (all types, per DMC): **Rs. 2,000** (urgent Rs. 4,000) - as fetched on 2026-09-29
- Re-checking of result (per subject): **Rs. 3,500** - as fetched on 2026-09-29
- Review of paper (per subject): **Rs. 7,000** - as fetched on 2026-09-29

**Documents to attach:**
- [ ] UHS Registration No. and father's name
- [ ] Principal / HOD Approval Form for Transcript (downloadable on the portal)
- [ ] Paid fee voucher

**Steps:**
1. On https://dip.uhs.edu.pk/ choose Transcript Issuance, Transcript with Verification, or Transcript Verification.
2. Get the Principal/HOD approval form signed, pay the voucher and upload it; track with Case Number + CNIC.

_Sources: https://dip.uhs.edu.pk/, https://www.uhs.edu.pk/fee-struct.php_

### Versioned fees - the same question on different dates

In [7]:
from datetime import date
fee = next(f for f in kb.services["must.degree"]["fees"] if f["item"].startswith("Degree before convocation"))
pd.DataFrame([{"question date": d, "fee in force": kb.fee_version(fee, date.fromisoformat(d))["amount"],
               "effective from": kb.fee_version(fee, date.fromisoformat(d))["effective_from"]}
              for d in ["2025-01-15", "2026-07-27", "2026-07-28", "2026-09-29"]])

,question date,fee in force,effective from
0,2025-01-15,7500,2024-09-02
1,2026-07-27,7500,2024-09-02
2,2026-07-28,9080,2026-07-28
3,2026-09-29,9080,2026-07-28


## 3. Admission advisor - students after 12 years of education

In [8]:
def advice_table(profile, n=12):
    r = advisor.advise(profile)
    rows = [{"program": p["program"][:60], "inst": p["institution"].upper(), "eligibility": p["eligibility"], "chance": p["chance"],
             "tests": ", ".join(t["id"] for t in p["tests"]) or "-",
             "1st sem fee": (p["fees"] or {}).get("first_semester"), "aggregate": p["merit_aggregate"]} for p in r["programs"]]
    return pd.DataFrame(rows).head(n)

print("Sample A: FSc pre-medical 82%, matric 88%, MDCAT 71%, wants medicine / health")
advice_table({"group": "pre_med", "hssc_percent": 82, "ssc_percent": 88, "mdcat_percent": 71, "interests": ["medicine", "health"]})

Sample A: FSc pre-medical 82%, matric 88%, MDCAT 71%, wants medicine / health


,program,inst,eligibility,chance,tests,1st sem fee,aggregate
0,"BS Allied Health Sciences / Medical Technology (Anesthesia,",UAJK,eligible,strong (heuristic),uajk_entry_test,79920.0,NaN
1,BS Microbiology,MUST,eligible,strong (heuristic),-,NaN,NaN
2,"Doctor of Physical Therapy (DPT, 5 years)",UAJK,eligible,good (heuristic),uajk_entry_test,118090.0,NaN
3,Pharm-D,UAJK,eligible,good (heuristic),uajk_entry_test,118090.0,NaN
4,Doctor of Pharmacy (Pharm-D) / Doctor of Physical Therapy /,MUST,eligible,good (heuristic),-,NaN,NaN
5,MBBS - AJK Medical College Muzaffarabad / MBBS Medical Colle,JAC,eligible,unknown - no verified closing merit; your aggregate is 77.1,mdcat,NaN,77.1
6,BS Nursing (School of Nursing),UAJK,check_required,unknown - minimum marks not published,uajk_entry_test,95320.0,NaN


In [9]:
print("Sample B: ICS 64%, wants computing")
advice_table({"group": "ics", "hssc_percent": 64, "interests": ["computing"]})

Sample B: ICS 64%, wants computing


,program,inst,eligibility,chance,tests,1st sem fee,aggregate
0,BS Computer Science / BS Artificial Intelligence,UAJK,eligible,moderate (heuristic),uajk_entry_test,77830,None
1,BS Cyber Security,UAJK,eligible,moderate (heuristic),uajk_entry_test,78110,None
2,BS Computer Science (Jhelum Valley or Neelum Campus),UAJK,eligible,good (heuristic),uajk_entry_test,71830,None
3,BS Computer Science / BS Information Technology,MUST,eligible,moderate (heuristic),-,59970,None
4,B.Sc. Software Engineering (PEC),UAJK,eligible,low to moderate (heuristic),uajk_entry_test,78830,None


In [10]:
print("Sample C: FA 58%, wants law / social sciences / business, only MUST")
advice_table({"group": "fa", "hssc_percent": 58, "interests": ["law", "social_sciences", "business"], "institutions": ["must"]})

Sample C: FA 58%, wants law / social sciences / business, only MUST


,program,inst,eligibility,chance,tests,1st sem fee,aggregate
0,BS Economics / Islamic Studies / International Relations / S,MUST,eligible,good (heuristic),-,45460.0,None
1,BS English,MUST,eligible,good (heuristic),-,49590.0,None
2,B.A. LLB (5 years),MUST,eligible,good (heuristic),hec_lat,50540.0,None
3,BBA / BS Accounting & Finance / BS Banking & Finance / BS Co,MUST,eligible,good (heuristic),-,NaN,None


In [11]:
print("Sample D: pre-engineering 56% wants engineering (below the 60% minimum) - shows why not eligible")
r = advisor.advise({"group": "pre_eng", "hssc_percent": 56, "interests": ["engineering"]})
pd.DataFrame([{"program": p["program"][:55], "eligibility": p["eligibility"], "reason": "; ".join(p["reasons"] + p["checks"])[:120]} for p in r["programs"]])

Sample D: pre-engineering 56% wants engineering (below the 60% minimum) - shows why not eligible


,program,eligibility,reason
0,B.Sc. Electrical Engineering (PEC),not_eligible,Minimum 60% in HSSC required; you have 56.0%.
1,B.Sc. Engineering - Electrical / Mechanical / Computer,not_eligible,Minimum 60% in HSSC required; you have 56.0%.; 60% + at least 33% in entry test by a P...
2,B.Sc. Civil Engineering,not_eligible,Minimum 60% in HSSC required; you have 56.0%.


In [12]:
print("Upcoming admission deadlines on 2026-09-29:")
pd.DataFrame(advisor.upcoming_deadlines(kb, date(2026, 9, 29), audience="applicants"))[["date", "days_left", "institution", "title", "trust"]]

Upcoming admission deadlines on 2026-09-29:


,date,days_left,institution,title,trust
0,2026-09-30,1,must,MUST 3rd merit list,official
1,2026-10-05,6,uajk,UAJK Fall 2026 admissions - last date to apply (extended),official
2,2026-10-05,6,uajk,UAJK merit scholarship / tuition fee rebate 2026-27 - apply by,official


## 4. Semester promotion engine (MUST rules, rule set `must.semester.v2017`)

In [13]:
def c(code, pct, credits=3, att=None): return {"code": code, "credits": credits, "percent": pct, "attendance": att}
ok, fail = [c("A", 65), c("B", 70), c("C", 55), c("D", 60)], [c("A", 40), c("B", 45), c("C", 62), c("D", 30)]
cases = {
    "all passed":                     [ok, ok],
    "fail sem 1 -> probation":        [fail],
    "fail sem 1 and 2 -> dropped":    [fail, fail],
    "summer semester waives":         [{"courses": fail, "summer": [c("A", 60), c("B", 58)]}],
    "probation then fail sem 3":      [fail, ok, fail],
    "fail sem 4, fail sem 5 (reset)": [ok, ok, ok, fail, fail],
    "fail sem 5 and 6 -> relegated":  [ok, ok, ok, ok, fail, fail],
}
rows = []
for name, sems in cases.items():
    s = {"institution": "must", "program_type": "bs4",
         "semesters": [{"number": i + 1, **(x if isinstance(x, dict) else {"courses": x})} for i, x in enumerate(sems)]}
    ev = progression.evaluate(s)
    rows.append({"case": name, "outcomes": " -> ".join(f"S{o['semester']}:{o['outcome']}" + (f"(to {o['relegate_to']})" if o["relegate_to"] else "") for o in ev["semesters"]),
                 "status": ev["state"]["status"], "probations used": ev["state"]["probations_used"], "CGPA": ev["cgpa"]["gpa"]})
pd.DataFrame(rows)

,case,outcomes,status,probations used,CGPA
0,all passed,S1:promoted -> S2:promoted,active,0,2.6
1,fail sem 1 -> probation,S1:probation,probation,1,2.6
2,fail sem 1 and 2 -> dropped,S1:probation -> S2:dropped,dropped,1,2.6
3,summer semester waives,S1:promoted,active,0,2.5
4,probation then fail sem 3,S1:probation -> S2:promoted -> S3:relegated(to 3),relegated,1,2.6
5,"fail sem 4, fail sem 5 (reset)",S1:promoted -> S2:promoted -> S3:promoted -> S4:probation -> S5:probation,probation,2,2.6
6,fail sem 5 and 6 -> relegated,S1:promoted -> S2:promoted -> S3:promoted -> S4:promoted -> S5:probation -> S6:relegat...,relegated,1,2.6


In [14]:
print("MUST's published worked examples:")
sem1 = [c("a", 52, 4), c("b", 70), c("c", 54, 2), c("d", 80, 1), c("e", 75)]
print(" semester example:", progression.gpa_report(sem1))
s = {"semesters": [{"number": 1, "courses": sem1}, {"number": 2, "courses": [c("f", 52, 4), c("g", 50), c("h", 50, 1)]}]}
print(" CGPA example    :", progression.gpa_report(progression.all_courses(s)))

MUST's published worked examples:
 semester example: {'weighted_percent': 63.92, 'gpa': 2.65, 'letter': 'C+', 'conflict': "Weighted average 63.92: MUST's Grading System page implies GP 2.65 (no rounding) but its Semester Rules example would give 2.70. Confirm with the Controller of Examinations."}
 CGPA example    : {'weighted_percent': 59.0, 'gpa': 2.45, 'letter': 'C', 'conflict': None}


## 5. What-if simulation (student S-001 has pending semester-3 results and is on probation)

In [15]:
s1 = store.get("students", "S-001")
last = progression.evaluate(s1)["semesters"][-1]
print("Pending:", last["pending"], "| passed", last["passed"], "of", last["opted"], "| need", last["required"])
rows = []
for scenario in [{"CS-301": 60, "CS-303": 62}, {"CS-301": 60, "CS-303": 40}, {"CS-301": 45, "CS-303": 45}]:
    w = progression.what_if(s1, scenario)["semesters"][-1]
    rows.append({"scenario": scenario, "outcome": w["outcome"], "relegate to": w["relegate_to"], "notes": " ".join(w["notes"])[:110]})
pd.DataFrame(rows)

Pending: ['CS-301', 'CS-303'] | passed 1 of 5 | need 3


,scenario,outcome,relegate to,notes
0,"{'CS-301': 60, 'CS-303': 62}",promoted,NaN,Promoted; the earlier probation still stands until the end of this half of the program.
1,"{'CS-301': 60, 'CS-303': 40}",relegated,3.0,Relegated to semester 3 (failed the threshold while on probation).
2,"{'CS-301': 45, 'CS-303': 45}",relegated,3.0,Relegated to semester 3 (failed the threshold while on probation).


## 6. Proactive dashboards - every finding routed to an action

In [16]:
def dash(sid):
    f = triage(monitor.scan(store.get("students", sid)), sid, record=False)
    return pd.DataFrame([{"action": x["action"], "finding": x["title"], "days left": x.get("days_left"), "detail": x["detail"][:100]} for x in f])

print("S-001 (MUST BS CS, on probation):"); display(dash("S-001"))
print("S-003 (UAJK, fee inside the 10% fine window):"); display(dash("S-003"))
print("S-002 (MUST final semester):"); display(dash("S-002"))
print("A-001 (applicant):"); display(dash("A-001"))

S-001 (MUST BS CS, on probation):


,action,finding,days left,detail
0,ESCALATE,Attendance below 75% in CS-302,NaN,Your attendance is 72%. Under MUST rules a student below 75% (lectures and practicals ...
1,ALERT,Hostel charges is overdue,-9.0,"Rs. 9,000 was due on 20 Sep 2026. Late fees or blocked registration may apply."
2,ALERT,Semester 4 dues (Spring-2027) due in 4 day(s),4.0,"Rs. 42,235 due on 03 Oct 2026."
3,ALERT,Pay next-semester dues before pre-registration,4.0,Pre-registration is 2 days before the semester (03 Oct 2026) and requires the dues to ...
4,ALERT,Semester 3 results pending,NaN,2 more of the 2 pending course(s) must be passed to meet the promotion threshold (3 of...
5,ALERT,You are on probation (probation 1 of the 2 allowed),NaN,The earlier probation stands until the end of this half of the program; another semest...
6,ALERT,Attendance close to the limit in CS-301,NaN,Attendance is 78%; below 75% you will be dropped from the course. No leave counts as r...
7,GUIDE,Re-checking window is open,8.0,"You failed CS-302, CS-201. Within 15 days of result declaration - apply by 07 Oct 2026..."


S-003 (UAJK, fee inside the 10% fine window):


,action,finding,days left,detail
0,ALERT,Semester 3 fee (BS English): 10% fine now applies,12.0,"Pay Rs. 63,470 (Rs. 57,700 + 10%) by 11 Oct 2026 (end of week 6)."
1,GUIDE,Promotion rules not available,NaN,UAJK has not published its semester promotion/probation rules online. Questions about ...


S-002 (MUST final semester):


,action,finding,days left,detail
0,GUIDE,You meet the degree requirements,None,CGPA 2.75 (65.24%). Next: department clearance certificate (11 offices + alumni portal...


A-001 (applicant):


,action,finding,days left,detail
0,ALERT,MUST 3rd merit list,1,Check https://admission.must.edu.pk/meritlist.php and deposit the admission fee after ...
1,ALERT,UAJK Fall 2026 admissions - last date to apply (extended),6,"Apply at https://admissions.uajk.edu.pk/ and pay the HBL challan (Rs. 3,000 + Rs. 1,00..."
2,ALERT,UAJK merit scholarship / tuition fee rebate 2026-27 - apply by,6,Top 3 in each undergraduate program get 50% / 40% / 30% tuition rebate


## 7. Event-driven monitoring - paying the overdue hostel dues clears the alert

In [17]:
s1 = store.get("students", "S-001")
before = {f["code"] for f in monitor.scan(s1)}
next(d for d in s1["dues"] if d["id"] == "D2").update(paid=True, paid_on="2026-09-29")
store.upsert("students", s1)
after = {f["code"] for f in monitor.scan(s1)}
print("removed:", before - after, "| added:", after - before)

removed: {'dues.overdue'} | added: set()


## 8. Application tracking - UAJK migration (submit within 15 days of paying the fee)

In [18]:
app = applications.create("uajk.migration", "S-003")
print(app["id"], app["status"], "| fees:", [(f["item"], f["amount"]) for f in app["fees"]])
app = applications.update(app["id"], checklist_done=[0, 1, 2, 3], fee_paid_on="2026-09-20")
print("status:", app["status"], "| deadlines:", applications.deadlines_for(app))

APP-0001 draft | fees: [('Migration certificate (ordinary, within 2 weeks)', 5800), ('Duplicate migration certificate (within 2 weeks)', 17700)]
status: fee_paid | deadlines: [{'rule': 'Submit the case within 15 days after depositing the fee', 'due_date': '2026-10-05', 'days_left': 6}]


## 9. Chat pipeline - sample questions

In [19]:
questions = [
    ("How do I get my degree from MUST?", None),
    ("what is the degree fee?", None),
    ("UAJK bonafide certificate", None),
    ("MUST transcript fee", None),
    ("mujhe MUST se clearance certificate kaise milega", None),
    ("UAJK duplicate degree lost degree", None),
    ("I did FSc pre-medical with 82%, matric 88%, MDCAT 71%, I want MBBS or DPT", None),
    ("ICS 64% which programs can I apply for in computing?", None),
    ("UAJK admission last date", None),
    ("Am I on probation? What happens if I fail again?", "S-001"),
    ("Can I get an extension for my fee deadline?", "S-003"),
    ("I am being harassed by a senior in hostel", "S-001"),
    ("MBBS degree from UHS", "S-004"),
]
rows = []
for q, sid in questions:
    r = assistant.answer(q, sid)
    rows.append({"question": q, "as": sid or "guest", "intent": r["intent"], "institution": r["institution"], "action": r["action"],
                 "first line": r["answer"].splitlines()[0][:80]})
pd.DataFrame(rows)

,question,as,intent,institution,action,first line
0,How do I get my degree from MUST?,guest,service,must,GUIDE,### Degree issuance (original degree) - Mirpur University of Science and Technol
1,what is the degree fee?,guest,information,NaN,ASK_INFO,Which institution is this for? **UAJK** - The University of Azad Jammu & Kashmir
2,UAJK bonafide certificate,guest,service,uajk,GUIDE,### Bonafide certificate - The University of Azad Jammu & Kashmir (UAJK)
3,MUST transcript fee,guest,service,must,GUIDE,"### Transcript / DMC (semester and final), duplicate and verification - Mirpur U"
4,mujhe MUST se clearance certificate kaise milega,guest,service,must,GUIDE,### Clearance certificate (no-dues) for university students - Mirpur University
5,UAJK duplicate degree lost degree,guest,service,uajk,GUIDE,### Duplicate degree (lost / damaged degree) - The University of Azad Jammu & Ka
6,"I did FSc pre-medical with 82%, matric 88%, MDCAT 71%, I want MBBS or DPT",guest,admission_advice,jac,GUIDE,"### Programs you can apply for (group: pre_med, HSSC 82.0%)"
7,ICS 64% which programs can I apply for in computing?,guest,admission_advice,NaN,GUIDE,"### Programs you can apply for (group: ics, HSSC 64.0%)"
8,UAJK admission last date,guest,deadlines,uajk,ALERT,### Deadlines
9,Am I on probation? What happens if I fail again?,S-001,student_status,must,ESCALATE,### Your status


In [20]:
display(Markdown(assistant.answer("I did FSc pre-medical with 82%, matric 88%, MDCAT 71%, I want MBBS or DPT")["answer"]))

### Programs you can apply for (group: pre_med, HSSC 82.0%)
- ✅ **BS Allied Health Sciences / Medical Technology (Anesthesia, Cardiovascular, Clinical Lab Sciences, Emergency, Surgery & OT, Radiology & Imaging, Optometry, Dialysis, Respiratory) / BS Public Health** (UAJK, Chellah Muzaffarabad) - chance: strong (heuristic); test: UAJK Entry Test - 1st semester Rs. 79,920 (2025-29)
    - Pre-Medical / Medical Technology, 2nd division
- ✅ **BS Microbiology** (MUST, Mirpur) - chance: strong (heuristic); test: no entry test listed
- ✅ **Doctor of Physical Therapy (DPT, 5 years)** (UAJK, Chellah Muzaffarabad) - chance: good (heuristic); test: UAJK Entry Test - 1st semester Rs. 118,090 (2025-30)
- ✅ **Pharm-D** (UAJK, Chellah Muzaffarabad) - chance: good (heuristic); test: UAJK Entry Test - 1st semester Rs. 118,090 (2025-30)
    - UAJK eligibility page says "similar to DPT (criteria not specified)" - confirm the program is offered this cycle
- ✅ **Doctor of Pharmacy (Pharm-D) / Doctor of Physical Therapy / BS Human Nutrition & Dietetics / BS Medical Lab Technology** (MUST, Mirpur) - chance: good (heuristic); test: no entry test listed
- ✅ **MBBS - AJK Medical College Muzaffarabad / MBBS Medical College Mirpur / Poonch Medical College Rawalakot** (JAC, Muzaffarabad / Mirpur / Rawalakot) - chance: unknown - no verified closing merit; your aggregate is 77.1; test: MDCAT (Medical & Dental Colleges Admission Test, conducted under PM&DC)
    - Fees for public-sector open-merit seats are not published on the college sites; MBBSMC's site shows a template-like fee table - ask the college. Degree awarded by the affiliating university (UHS for PMC).
- ⚠️ **BS Nursing (School of Nursing)** (UAJK, Chellah Muzaffarabad) - chance: unknown - minimum marks not published; test: UAJK Entry Test - 1st semester Rs. 95,320 (2025-29)
    - Minimum marks are not published - confirm with the admission office.
    - Eligibility not published on the UAJK eligibility page - nursing programs follow PNC criteria; confirm with the School of Nursing

**How to apply - Joint Admission Committee (JAC) AJ&K - MBBS/BDS admissions for the public medical & dental colleges of AJK** (https://jac.ajk.gov.pk/)
1. Pass MDCAT (MBBS 55%, BDS 50%) in your domicile region.
2. Apply ONLINE on the JAC AJ&K portal (no paper applications) - register with email, fill the profile, upload scanned documents, set college preferences (they cannot be changed after submission).
3. Pay the processing fee (reported as Rs. 3,000 + Rs. 400 IT charges via Easy Paisa / debit card / online banking).
4. Follow the provisional merit list and selection lists notified by the JAC Muzaffarabad secretariat.
Documents: Secondary School Certificate (SSC) and result card; HSSC / F.Sc. (Pre-Medical) certificate and result card (A-level needs an IBCC equivalence certificate); Official MDCAT result card (valid for the current session); State Subject certificate (Class-I) and domicile (AJ&K); Permanent Resident Certificate (PRC); Candidate's CNIC / Form-B and guardian's CNIC; 8 passport-size photographs with blue background; Different-abilities certificate (if applicable)

**How to apply - Mirpur University of Science and Technology (MUST)** (https://admission.must.edu.pk/)
1. Apply online at https://admission.must.edu.pk/ and register for SEET (http://seet.must.edu.pk/) where an entry test is required.
2. Choose a category - Subsidized (merit-based, normal fee) or Open Merit (self-financing, pays the additional non-subsidized tuition fee).
3. Check merit lists at https://admission.must.edu.pk/meritlist.php.
4. After selection, the admission committee chairperson verifies ORIGINAL documents; then deposit the admission fee through the bank challan.
Documents: Attested academic certificates / result cards (SSC, HSSC); NOC if F.A./F.Sc. (or equivalent) was passed from a board other than BISE Mirpur AJ&K; CNIC / Form-B and domicile/State Subject (confirm with the current advertisement); New Student's Undertaking (affidavit) - https://must.edu.pk/wp-content/uploads/2025/09/newstudentUndertaking-1.pdf; Medical certificate - https://must.edu.pk/wp-content/uploads/2025/09/medical_certificate-1.pdf

**How to apply - The University of Azad Jammu & Kashmir (UAJK)** (https://admissions.uajk.edu.pk/)
1. Apply online at https://admissions.uajk.edu.pk/ (or collect a hard-copy form from the DSA office, Chellah Campus).
2. Generate the online challan - Rs. 3,000 for the first program, plus Rs. 1,000 per additional program; entry test fee Rs. 1,000 + Rs. 200 per additional program is added automatically. Pay at any HBL branch.
3. Payment against the FIRST challan is mandatory - an application is not accepted if only the 2nd/3rd challans are paid.
4. Online applicants must also send printed copies of each application, with the challan copy and documents, to the Directorate of Students' Affairs (or their campus).
5. Appear in the entry test at your campus (where applicable). Merit lists are displayed on DSA / department notice boards and the website; no admission letter is issued.
6. Bring ORIGINAL documents for verification at the interview.
Documents: Attested copies of all educational certificates and degrees; Character certificate from the college/institute last attended; NOC / migration certificate if HSSC is from a board other than BISE Mirpur AJ&K; Copy of Domicile and State Subject certificate; Copy of CNIC or Form-B; Four recent passport-size photographs; Copy of guardian's CNIC; NOC from employer (government/other employees); study leave certificate for full-time study; Relevant nomination/certificate for reserved seats (see reserved_seats)

**Upcoming deadlines:**
- 2026-09-30 (1 days): MUST 3rd merit list
- 2026-10-05 (6 days): UAJK Fall 2026 admissions - last date to apply (extended)
- 2026-10-05 (6 days): UAJK merit scholarship / tuition fee rebate 2026-27 - apply by

_Chance bands are a heuristic based on your margin over the published minimum and program demand. Actual selection depends on the entry test, the merit formula and closing merit of the year, which the institutions do not publish in a verifiable form._

## 10. Evidence search and the human-advisor queue

In [21]:
pd.DataFrame([{"score": round(s, 1), "kind": ch.kind, "trust": ch.trust, "source": ch.source, "text": ch.text[:90].replace("\n", " ")}
              for s, ch in kb.search("rechecking answer book fee days", k=6)])

,score,kind,trust,source,text
0,45.3,service,official,https://uajk.edu.pk/wp-content/uploads/2025/10/Rechecking-Form.pdf,Re-checking of answer book / result (The University of Azad Jammu & Kashmir (UAJK)) id...
1,43.5,service,official,https://must.edu.pk/wp-content/uploads/2022/10/Paper-Rechecking-Application-1.pdf,Re-checking of answer book / result (Mirpur University of Science and Technology (MUST...
2,23.5,document,official,documents/must/text/Paper_Rechecking_Application.txt,"MIRPUR UNIVERSITY OF SCIENCE & TECHNOLOGY (MUST), MIRPUR AZAD JAMMU & KASHMIR CONTRO..."
3,21.5,document,official,documents/uajk/text/Application_Form_for_Re_Checking_of_the_Answer_Book_Result.txt,_______________ Full Address:________________ __________________________ __________...
4,21.1,document,official,documents/uajk/text/Application_Form_for_Re_Checking_of_the_Answer_Book_Result.txt,The University of Azad Jammu & Kashmir Muzaffarabad (Examinations Department) Applic...
5,17.6,document,official,documents/must/text/Paper_Rechecking_Application.txt,__________ 10. Amount Paid:_________________ 11. Bank receipt No. and date of fee pai...


In [22]:
pd.DataFrame(store.load("escalations"))[["id", "student_id", "summary", "status"]]

,id,student_id,summary,status
0,ESC-0001,S-001,Attendance below 75% in CS-302,open
1,ESC-0002,S-003,Can I get an extension for my fee deadline?,open
2,ESC-0003,S-001,I am being harassed by a senior in hostel,open


## 11. The MCP server, through the official MCP client (stdio)

In [23]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def mcp_demo():
    params = StdioServerParameters(command=sys.executable, args=[str(ROOT / "mcp_server" / "server.py")],
                                   env={**os.environ, "JK_DISABLE_LLM": "1", "PYTHONIOENCODING": "utf-8"})
    # Jupyter's sys.stderr has no OS file handle, so send the server's stderr log to a real file
    with open(Path(os.environ["JK_DATA_DIR"]) / "mcp_server.log", "w") as errlog:
        async with stdio_client(params, errlog=errlog) as (r, w):
            async with ClientSession(r, w) as s:
                await s.initialize()
                print("tools:", [t.name for t in (await s.list_tools()).tools])
                res = await s.call_tool("get_service", {"service_id": "uajk.english_proficiency"})
                card = json.loads(res.content[0].text)
                print(card["name"], "|", [(f["item"], f["amount"]) for f in card["fees"]], "|", card["processing_time"])

await mcp_demo()

tools: ['list_services', 'get_service', 'search_knowledge', 'advise_admission', 'get_deadlines', 'evaluate_progression', 'student_dashboard', 'start_application', 'ask_jk_assistance']
English proficiency certificate (medium of instruction letter) | [('English proficiency certificate (non-refundable)', 3900)] | 7 working days


## 12. (Optional) Claude-written answer

With `ANTHROPIC_API_KEY` set before starting Jupyter, this cell calls `claude-opus-5-5`. Claude may only use the rule-engine result and the numbered evidence. Without a key it is skipped.

In [24]:
if USE_CLAUDE:
    os.environ.pop("JK_DISABLE_LLM", None)
    r = assistant.answer("What documents do I need for a UAJK duplicate degree and how much does it cost?")
    display(Markdown(r["answer"]))
else:
    print("Skipped - set ANTHROPIC_API_KEY to see Claude-composed answers.")

Skipped - set ANTHROPIC_API_KEY to see Claude-composed answers.
